# CosyVoice Inference Walkthrough

**Goal**: Walk through the full CosyVoice TTS inference pipeline step by step, showing concrete tensor shapes at every stage. No actual model weights needed — we use dummy tensors to illustrate the data flow.

CosyVoice is a **hybrid** TTS system (Paradigm 1 + Paradigm 2 combined):
- **Stage 1**: Autoregressive LLM generates coarse *speech tokens* ("what to say and how")
- **Stage 2**: Flow-matching generates fine *mel spectrogram* ("exactly how it sounds")
- **Stage 3**: Vocoder converts mel → waveform

Think of it as: **writing sheet music** (Stage 1) → **performing it** (Stage 2) → **recording it** (Stage 3).

```
"Hello world" ──→ [G2P] ──→ [Text Encoder] ──→ [AR LLM] ──→ [Flow Matching] ──→ [Vocoder] ──→ 🔊
  string         phoneme IDs   vectors        speech tokens   mel spectrogram    waveform
```

In [1]:
import torch
import torch.nn as nn
import numpy as np

# Key constants from CosyVoice
SAMPLE_RATE = 24_000        # output audio sample rate
MEL_BINS = 80               # number of mel frequency bins
MEL_HOP = 480               # samples per mel frame (24000 / 50 = 480)
MEL_FRAME_RATE = 50         # mel frames per second
TOKEN_RATE = 25             # speech tokens per second (1 token = 2 mel frames)
CODEBOOK_SIZE = 6561        # FSQ codebook size in CosyVoice 2
LLM_HIDDEN_DIM = 896        # Qwen2.5-0.5B hidden dimension
SPK_EMBED_DIM = 192         # CAMPPlus speaker embedding dimension

print(f"Key rates:")
print(f"  1 second of audio = {SAMPLE_RATE:,} samples")
print(f"  1 second of audio = {MEL_FRAME_RATE} mel frames")
print(f"  1 second of audio = {TOKEN_RATE} speech tokens")
print(f"  1 speech token = {MEL_FRAME_RATE // TOKEN_RATE} mel frames")
print(f"  1 mel frame = {MEL_HOP} audio samples")

Key rates:
  1 second of audio = 24,000 samples
  1 second of audio = 50 mel frames
  1 second of audio = 25 speech tokens
  1 speech token = 2 mel frames
  1 mel frame = 480 audio samples


## Step 0: Frontend — Prepare All Inputs

Before any neural network runs, the frontend prepares raw inputs into tensors.

There are **two inputs** to CosyVoice:
1. **Text** — what you want the model to say (e.g., "Hello world")
2. **Reference audio** — a 3-10 second clip of the voice you want to clone

The frontend processes both:

In [2]:
# ── Input text ──────────────────────────────────────────────
text = "Hello world, this is a test."

# Step 0a: G2P tokenizer (rule-based, NOT a neural network)
# Converts string → integer IDs
# CosyVoice 2 uses BPE tokenizer (same as Qwen2.5)
text_token_ids = torch.tensor([[9707, 1879, 11, 419, 374, 264, 1273, 13]])  # dummy BPE IDs
T_text = text_token_ids.shape[1]

print(f"Input text: '{text}'")
print(f"After tokenizer: {text_token_ids}")
print(f"Shape: {text_token_ids.shape}  →  (batch=1, T_text={T_text})")
print(f"Each integer is a BPE token ID — just a lookup index, nothing neural yet.")

Input text: 'Hello world, this is a test.'
After tokenizer: tensor([[9707, 1879,   11,  419,  374,  264, 1273,   13]])
Shape: torch.Size([1, 8])  →  (batch=1, T_text=8)
Each integer is a BPE token ID — just a lookup index, nothing neural yet.


In [3]:
# ── Reference audio (for voice cloning) ────────────────────
prompt_duration_sec = 3.0  # 3-second reference clip

# Step 0b: Extract speech tokens from reference audio
# Uses the S3 tokenizer (a separate ONNX model: audio → mel → encoder → VQ → integer tokens)
T_prompt_tokens = int(prompt_duration_sec * TOKEN_RATE)  # 3s * 25 = 75 tokens
prompt_speech_tokens = torch.randint(0, CODEBOOK_SIZE, (1, T_prompt_tokens))

print(f"Reference audio: {prompt_duration_sec}s clip")
print(f"After S3 tokenizer: shape {prompt_speech_tokens.shape}  →  (1, {T_prompt_tokens})")
print(f"Each integer ∈ [0, {CODEBOOK_SIZE}) represents ~{1000/TOKEN_RATE:.0f}ms of speech")
print(f"Sample tokens: {prompt_speech_tokens[0, :8].tolist()} ...")

Reference audio: 3.0s clip
After S3 tokenizer: shape torch.Size([1, 75])  →  (1, 75)
Each integer ∈ [0, 6561) represents ~40ms of speech
Sample tokens: [5561, 904, 292, 5690, 2532, 6488, 3392, 2634] ...


In [4]:
# Step 0c: Extract mel spectrogram from reference audio (needed later for flow-matching)
T_prompt_mel = int(prompt_duration_sec * MEL_FRAME_RATE)  # 3s * 50 = 150 frames
prompt_mel = torch.randn(1, MEL_BINS, T_prompt_mel)  # dummy mel

print(f"Reference mel spectrogram: {prompt_mel.shape}  →  (1, {MEL_BINS} freq bins, {T_prompt_mel} time frames)")
print(f"Think of it as a {MEL_BINS}×{T_prompt_mel} grayscale image (frequency × time)")

Reference mel spectrogram: torch.Size([1, 80, 150])  →  (1, 80 freq bins, 150 time frames)
Think of it as a 80×150 grayscale image (frequency × time)


In [5]:
# Step 0d: Extract speaker embedding from reference audio
# Uses CAMPPlus model (a speaker verification network) — outputs a single fixed-size vector
spk_embedding = torch.randn(1, SPK_EMBED_DIM)
spk_embedding = spk_embedding / spk_embedding.norm()  # L2 normalized

print(f"Speaker embedding: {spk_embedding.shape}  →  (1, {SPK_EMBED_DIM})")
print(f"This is ONE vector that captures 'who this person is' — their vocal identity.")
print(f"Same reference audio always → same embedding. Different speakers → different embeddings.")

Speaker embedding: torch.Size([1, 192])  →  (1, 192)
This is ONE vector that captures 'who this person is' — their vocal identity.
Same reference audio always → same embedding. Different speakers → different embeddings.


In [6]:
# ── Summary of all frontend outputs ────────────────────────
print("=" * 60)
print("FRONTEND OUTPUTS (inputs to neural pipeline)")
print("=" * 60)
print(f"  text_token_ids:       {text_token_ids.shape}        ← what to say")
print(f"  prompt_speech_tokens: {prompt_speech_tokens.shape}       ← reference voice as tokens")
print(f"  prompt_mel:           {prompt_mel.shape}  ← reference voice as mel")
print(f"  spk_embedding:        {spk_embedding.shape}       ← speaker identity vector")
print()
print("All integers and floats. Ready for the neural networks.")

FRONTEND OUTPUTS (inputs to neural pipeline)
  text_token_ids:       torch.Size([1, 8])        ← what to say
  prompt_speech_tokens: torch.Size([1, 75])       ← reference voice as tokens
  prompt_mel:           torch.Size([1, 80, 150])  ← reference voice as mel
  spk_embedding:        torch.Size([1, 192])       ← speaker identity vector

All integers and floats. Ready for the neural networks.


## Stage 1: LLM — Text → Speech Tokens

This is the **autoregressive** stage. It uses a **Qwen2.5-0.5B** language model (same family as a chat LLM!) to predict speech tokens one at a time.

The key insight: **speech tokens are just another language**. The LLM was trained on both text and speech tokens, so it learned to "translate" from text to speech tokens.

### Input Sequence Construction

Just like GPT takes a prompt, CosyVoice's LLM takes a carefully constructed input sequence:

```
[SOS] [text tokens] [task token] [prompt speech tokens] → generate new speech tokens...
```

In [7]:
# The LLM has TWO embedding tables:
#   1. text_embedding:   text token ID → vector (shared with Qwen2.5's word embeddings)
#   2. speech_embedding: speech token ID → vector (separate, learned for speech)

text_embed_table = nn.Embedding(151936, LLM_HIDDEN_DIM)    # Qwen2.5 vocab size
speech_embed_table = nn.Embedding(CODEBOOK_SIZE + 3, LLM_HIDDEN_DIM)  # +3 for special tokens (SOS, EOS, etc.)

# Embed text tokens
text_embeddings = text_embed_table(text_token_ids)
print(f"Text tokens {text_token_ids.shape} → text embeddings {text_embeddings.shape}")
print(f"  Each integer became a {LLM_HIDDEN_DIM}-dim vector")

# Embed prompt speech tokens  
prompt_speech_embeddings = speech_embed_table(prompt_speech_tokens)
print(f"\nPrompt speech tokens {prompt_speech_tokens.shape} → speech embeddings {prompt_speech_embeddings.shape}")

Text tokens torch.Size([1, 8]) → text embeddings torch.Size([1, 8, 896])
  Each integer became a 896-dim vector

Prompt speech tokens torch.Size([1, 75]) → speech embeddings torch.Size([1, 75, 896])


In [8]:
# Construct the full input sequence by concatenating everything:
#   [SOS_embed | text_embeds | task_embed | prompt_speech_embeds]

sos_embed = torch.randn(1, 1, LLM_HIDDEN_DIM)   # start-of-sequence
task_embed = torch.randn(1, 1, LLM_HIDDEN_DIM)   # task indicator (zero-shot TTS)

lm_input = torch.cat([
    sos_embed,                    # (1, 1, 896)
    text_embeddings,              # (1, T_text, 896)
    task_embed,                   # (1, 1, 896)
    prompt_speech_embeddings,     # (1, T_prompt, 896)
], dim=1)

print(f"LLM input sequence: {lm_input.shape}")
print(f"  = 1 (SOS) + {T_text} (text) + 1 (task) + {T_prompt_tokens} (prompt speech)")
print(f"  = {lm_input.shape[1]} tokens total")
print(f"\nThis is exactly like giving GPT a prompt. The model reads this,")
print(f"then generates new speech tokens that continue the pattern.")

LLM input sequence: torch.Size([1, 85, 896])
  = 1 (SOS) + 8 (text) + 1 (task) + 75 (prompt speech)
  = 85 tokens total

This is exactly like giving GPT a prompt. The model reads this,
then generates new speech tokens that continue the pattern.


In [9]:
# ── Autoregressive generation (the slow part) ──────────────
# The LLM generates speech tokens ONE AT A TIME, like GPT generating words.

target_duration_sec = 2.0
T_gen_tokens = int(target_duration_sec * TOKEN_RATE)  # 2s * 25 = 50 tokens

# In real CosyVoice, each step does:
#   1. Feed current sequence into Qwen2.5 (with KV cache for speed)
#   2. Take the last hidden state → linear projection → logits over codebook
#   3. Sample from logits → next speech token
#   4. Append to sequence, repeat

llm_decoder_head = nn.Linear(LLM_HIDDEN_DIM, CODEBOOK_SIZE + 3)  # maps hidden → logits

# Simulate the loop (in practice this runs on GPU with KV cache)
generated_tokens = []
print(f"Generating {T_gen_tokens} speech tokens ({target_duration_sec}s of audio):")
for step in range(T_gen_tokens):
    # Simulate: LLM outputs a hidden state for the last position
    hidden_state = torch.randn(1, 1, LLM_HIDDEN_DIM)
    
    # Project to logits over the speech codebook
    logits = llm_decoder_head(hidden_state[:, -1, :])  # (1, 6564)
    
    # Sample (top-k, temperature, etc.)
    probs = torch.softmax(logits / 0.8, dim=-1)  # temperature=0.8
    token = torch.multinomial(probs, 1)  # (1, 1)
    generated_tokens.append(token.item())
    
    if step < 3 or step == T_gen_tokens - 1:
        print(f"  Step {step:3d}: hidden {hidden_state.shape} → logits {logits.shape} → token {token.item()}")
    elif step == 3:
        print(f"  ...")

generated_speech_tokens = torch.tensor([generated_tokens])
print(f"\nLLM output: {generated_speech_tokens.shape}  →  (1, {T_gen_tokens}) speech tokens")
print(f"These are just integers! They encode rhythm, intonation, phoneme identity.")
print(f"But they are NOT audio yet — they need to be 'rendered' by the flow-matching stage.")

Generating 50 speech tokens (2.0s of audio):
  Step   0: hidden torch.Size([1, 1, 896]) → logits torch.Size([1, 6564]) → token 1897
  Step   1: hidden torch.Size([1, 1, 896]) → logits torch.Size([1, 6564]) → token 3685
  Step   2: hidden torch.Size([1, 1, 896]) → logits torch.Size([1, 6564]) → token 2944
  ...
  Step  49: hidden torch.Size([1, 1, 896]) → logits torch.Size([1, 6564]) → token 367

LLM output: torch.Size([1, 50])  →  (1, 50) speech tokens
These are just integers! They encode rhythm, intonation, phoneme identity.
But they are NOT audio yet — they need to be 'rendered' by the flow-matching stage.


## Stage 2: Flow Matching — Speech Tokens → Mel Spectrogram

Now we have coarse speech tokens (integers). We need to turn them into a **mel spectrogram** — a detailed acoustic representation.

Flow matching does this by:
1. **Encode** tokens into continuous vectors (the conditioning signal μ)
2. **Start from pure noise** X₀ ~ N(0, I)
3. **Iteratively denoise** using a U-Net, guided by μ and speaker embedding
4. After ~10 steps, the noise has become a clean mel spectrogram

Analogy: Imagine you have a blurry photo (noise) and GPS directions (the speech tokens). Each step makes the photo slightly clearer, following the directions.

In [10]:
# ── Step 2a: Encode speech tokens → conditioning signal μ ──
# The flow model has its own token encoder (NOT the LLM — a separate smaller transformer)
# It takes the concatenated prompt + generated tokens and outputs mu

all_tokens = torch.cat([prompt_speech_tokens, generated_speech_tokens], dim=1)
T_all_tokens = all_tokens.shape[1]
print(f"Concatenated tokens: {all_tokens.shape}  →  (1, {T_prompt_tokens} prompt + {T_gen_tokens} generated = {T_all_tokens})")

# Token encoder: embed → transformer → project to mel-sized vectors
# Then upsample 2x (because token rate is 25Hz but mel rate is 50Hz)
T_all_mel = T_all_tokens * 2  # 2x upsample
mu = torch.randn(1, MEL_BINS, T_all_mel)  # conditioning signal

print(f"After token encoder + 2x upsample:")
print(f"  μ (conditioning): {mu.shape}  →  (1, {MEL_BINS} freq bins, {T_all_mel} mel frames)")
print(f"")
print(f"  Why 2x upsample?")
print(f"    Token rate = {TOKEN_RATE} Hz  →  {T_all_tokens} tokens")
print(f"    Mel rate   = {MEL_FRAME_RATE} Hz  →  {T_all_mel} mel frames")
print(f"    Ratio: {MEL_FRAME_RATE}/{TOKEN_RATE} = {MEL_FRAME_RATE // TOKEN_RATE}x")

Concatenated tokens: torch.Size([1, 125])  →  (1, 75 prompt + 50 generated = 125)
After token encoder + 2x upsample:
  μ (conditioning): torch.Size([1, 80, 250])  →  (1, 80 freq bins, 250 mel frames)

  Why 2x upsample?
    Token rate = 25 Hz  →  125 tokens
    Mel rate   = 50 Hz  →  250 mel frames
    Ratio: 50/25 = 2x


In [11]:
# ── Step 2b: Set up the masked mel ─────────────────────────
# For the prompt region: use the REAL mel from the reference audio
# For the generation region: mask with zeros (this is what we want to generate)

T_gen_mel = T_gen_tokens * 2  # generation region in mel frames

# The flow model sees: [real prompt mel | zeros to fill in]
masked_mel = torch.cat([
    prompt_mel,                                    # (1, 80, 150) — real
    torch.zeros(1, MEL_BINS, T_gen_mel),           # (1, 80, 100) — to generate
], dim=2)

print(f"Masked mel: {masked_mel.shape}")
print(f"  Prompt region (real):     frames 0-{T_prompt_mel-1}")
print(f"  Generation region (zeros): frames {T_prompt_mel}-{T_prompt_mel + T_gen_mel - 1}")

Masked mel: torch.Size([1, 80, 250])
  Prompt region (real):     frames 0-149
  Generation region (zeros): frames 150-249


In [12]:
# ── Step 2c: ODE solver — iteratively denoise ─────────────
# Start from Gaussian noise, iteratively move toward the target mel
# Each step: the U-Net predicts a velocity field, and we step along it

NFE = 10  # number of function evaluations (ODE steps)

# Start from noise
x_t = torch.randn(1, MEL_BINS, T_gen_mel)  # (1, 80, 100) — pure noise

print(f"ODE solver: {NFE} steps, from noise to mel spectrogram")
print(f"Initial x_0 (noise): {x_t.shape}, mean={x_t.mean():.3f}, std={x_t.std():.3f}")
print()

# Simulate the ODE integration
# In real CosyVoice, the U-Net takes [x_t, mu, spk_embed, timestep] → velocity
for step in range(NFE):
    t = step / NFE  # time goes from 0 (noise) to 1 (clean mel)
    dt = 1.0 / NFE
    
    # In reality: velocity = UNet(x_t, mu, spk_embedding, t)
    # The U-Net input is [x_t concat mu] = (1, 160, T) along channel dim
    unet_input_channels = MEL_BINS * 2  # 80 (x_t) + 80 (mu) = 160
    
    # Simulate: velocity pushes x_t toward the target
    velocity = torch.randn_like(x_t) * (1 - t)  # velocity decreases as we approach target
    
    # Euler step: x_{t+dt} = x_t + dt * velocity
    x_t = x_t + dt * velocity
    
    if step < 2 or step >= NFE - 2:
        print(f"  Step {step}: t={t:.2f} → U-Net input channels={unet_input_channels} → velocity → x_t std={x_t.std():.3f}")
    elif step == 2:
        print(f"  ...")

generated_mel = x_t  # final output
print(f"\nFlow output: {generated_mel.shape}  →  mel spectrogram for {target_duration_sec}s of speech")
print(f"  {MEL_BINS} frequency bins × {T_gen_mel} time frames")

ODE solver: 10 steps, from noise to mel spectrogram
Initial x_0 (noise): torch.Size([1, 80, 100]), mean=0.008, std=1.006

  Step 0: t=0.00 → U-Net input channels=160 → velocity → x_t std=1.010
  Step 1: t=0.10 → U-Net input channels=160 → velocity → x_t std=1.014
  ...
  Step 8: t=0.80 → U-Net input channels=160 → velocity → x_t std=1.025
  Step 9: t=0.90 → U-Net input channels=160 → velocity → x_t std=1.025

Flow output: torch.Size([1, 80, 100])  →  mel spectrogram for 2.0s of speech
  80 frequency bins × 100 time frames


In [13]:
# ── What the U-Net actually sees at each step ──────────────
# Let's be very explicit about the U-Net's inputs:

print("U-Net inputs at each ODE step:")
print("=" * 60)

# 1. x_t: current noisy mel
print(f"  x_t (noisy mel):       (1, {MEL_BINS}, {T_gen_mel})   ← starts as noise, gets cleaner")

# 2. mu: conditioning from speech tokens (constant across steps)
mu_gen_region = mu[:, :, T_prompt_mel:]  # only the generation region
print(f"  μ (token conditioning): (1, {MEL_BINS}, {T_gen_mel})   ← from encoded speech tokens")

# Concatenated along channel dim:
print(f"  ────────────────────────────────────")
print(f"  concat([x_t, μ]):      (1, {MEL_BINS * 2}, {T_gen_mel})  ← actual U-Net input")

# 3. Speaker embedding: broadcast across time
print(f"  spk_embed:              (1, {SPK_EMBED_DIM})        ← broadcast & concatenated inside U-Net")

# 4. Timestep
print(f"  timestep t:             scalar ∈ [0, 1]     ← tells U-Net how noisy x_t is")

print(f"\nU-Net output:")
print(f"  velocity:              (1, {MEL_BINS}, {T_gen_mel})   ← direction to move x_t")

U-Net inputs at each ODE step:
  x_t (noisy mel):       (1, 80, 100)   ← starts as noise, gets cleaner
  μ (token conditioning): (1, 80, 100)   ← from encoded speech tokens
  ────────────────────────────────────
  concat([x_t, μ]):      (1, 160, 100)  ← actual U-Net input
  spk_embed:              (1, 192)        ← broadcast & concatenated inside U-Net
  timestep t:             scalar ∈ [0, 1]     ← tells U-Net how noisy x_t is

U-Net output:
  velocity:              (1, 80, 100)   ← direction to move x_t


## Stage 3: Vocoder — Mel Spectrogram → Waveform

The mel spectrogram is a compressed representation of audio (80 frequency bins × T frames). Now we need to convert it to an actual audio waveform — the signal that a speaker plays.

CosyVoice uses **HiFi-GAN** with a built-in F0 (pitch) predictor.

In [14]:
# ── Step 3: Vocoder ────────────────────────────────────────

# Input: mel spectrogram
print(f"Vocoder input:  {generated_mel.shape}  →  (1, {MEL_BINS}, {T_gen_mel})")

# Inside the vocoder:
#   1. F0 predictor: mel → pitch contour (fundamental frequency at each frame)
#   2. Source module: pitch → harmonic excitation signal (sine waves)
#   3. Upsampling convolutions: upsample from mel frame rate to audio sample rate
#   4. ISTFT: inverse short-time Fourier transform → waveform

N_samples = T_gen_mel * MEL_HOP  # 100 frames * 480 samples/frame = 48000 samples
waveform = torch.randn(1, N_samples).clamp(-0.99, 0.99)  # simulate output

print(f"Vocoder output: {waveform.shape}  →  (1, {N_samples:,})")
print(f"")
print(f"Duration: {N_samples / SAMPLE_RATE:.2f} seconds at {SAMPLE_RATE:,} Hz")
print(f"")
print(f"Math: {T_gen_mel} mel frames × {MEL_HOP} samples/frame = {N_samples:,} samples")
print(f"      {N_samples:,} samples ÷ {SAMPLE_RATE:,} Hz = {N_samples / SAMPLE_RATE:.2f} seconds")

Vocoder input:  torch.Size([1, 80, 100])  →  (1, 80, 100)
Vocoder output: torch.Size([1, 48000])  →  (1, 48,000)

Duration: 2.00 seconds at 24,000 Hz

Math: 100 mel frames × 480 samples/frame = 48,000 samples
      48,000 samples ÷ 24,000 Hz = 2.00 seconds


## Full Pipeline Summary

Let's trace the complete data flow for generating 2 seconds of speech from a 3-second reference clip:

In [15]:
print("CosyVoice Inference: Full Tensor Flow")
print("=" * 70)
print()
print("FRONTEND (no neural networks, just preprocessing)")
print("─" * 70)
print(f'  "Hello world"        string')
print(f'    → G2P tokenizer    → text_tokens:          (1, {T_text})          int')
print(f'  3s reference audio')
print(f'    → S3 tokenizer     → prompt_speech_tokens:  (1, {T_prompt_tokens})         int')
print(f'    → Mel extraction   → prompt_mel:            (1, {MEL_BINS}, {T_prompt_mel})      float')
print(f'    → CAMPPlus         → spk_embedding:         (1, {SPK_EMBED_DIM})        float')
print()
print("STAGE 1: LLM (autoregressive, Qwen2.5-0.5B)")
print("─" * 70)
print(f'  text_tokens + prompt_speech_tokens')
print(f'    → Embed            → (1, {1 + T_text + 1 + T_prompt_tokens}, {LLM_HIDDEN_DIM})            float vectors')
print(f'    → Transformer      → hidden states')
print(f'    → Linear head      → logits (1, {CODEBOOK_SIZE}+3) per step')
print(f'    → Sample           → speech_tokens:         (1, {T_gen_tokens})         int')
print(f'                          {T_gen_tokens} tokens × {1000//TOKEN_RATE}ms/token = {target_duration_sec}s')
print()
print("STAGE 2: FLOW MATCHING (iterative, 10 ODE steps)")
print("─" * 70)
print(f'  speech_tokens (prompt + generated)')
print(f'    → Token encoder    → μ:                     (1, {MEL_BINS}, {T_all_mel})      float (conditioning)')
print(f'  Start from noise:      x₀ ~ N(0,I):          (1, {MEL_BINS}, {T_gen_mel})      float')
print(f'    → U-Net × {NFE}       → velocity per step')
print(f'    → Euler integrate  → mel_spectrogram:       (1, {MEL_BINS}, {T_gen_mel})      float')
print()
print("STAGE 3: VOCODER (single forward pass, HiFi-GAN)")
print("─" * 70)
print(f'  mel_spectrogram')
print(f'    → F0 predictor     → pitch contour')
print(f'    → Upsample + ISTFT → waveform:              (1, {N_samples:,})      float')
print(f'                          {N_samples:,} samples ÷ {SAMPLE_RATE:,} Hz = {N_samples/SAMPLE_RATE:.1f}s')
print()
print("=" * 70)
print(f'Done! {target_duration_sec}s of cloned speech from a {prompt_duration_sec}s reference.')

CosyVoice Inference: Full Tensor Flow

FRONTEND (no neural networks, just preprocessing)
──────────────────────────────────────────────────────────────────────
  "Hello world"        string
    → G2P tokenizer    → text_tokens:          (1, 8)          int
  3s reference audio
    → S3 tokenizer     → prompt_speech_tokens:  (1, 75)         int
    → Mel extraction   → prompt_mel:            (1, 80, 150)      float
    → CAMPPlus         → spk_embedding:         (1, 192)        float

STAGE 1: LLM (autoregressive, Qwen2.5-0.5B)
──────────────────────────────────────────────────────────────────────
  text_tokens + prompt_speech_tokens
    → Embed            → (1, 85, 896)            float vectors
    → Transformer      → hidden states
    → Linear head      → logits (1, 6561+3) per step
    → Sample           → speech_tokens:         (1, 50)         int
                          50 tokens × 40ms/token = 2.0s

STAGE 2: FLOW MATCHING (iterative, 10 ODE steps)
──────────────────────────────

## Key Takeaways

| Question | Answer |
|---|---|
| What does the **G2P tokenizer** output? | Integer IDs (e.g., `[14, 3, 27, 42]`). Rule-based, not neural. |
| What does the **text encoder** output? | Continuous vectors, one per token, ~256-896 dims. Contextual — knows what's around each phoneme. |
| What does the **LLM** output? | Integer speech tokens. Each one = 40ms of sound. Generated one at a time (slow). |
| What does **flow matching** output? | Mel spectrogram — a 2D float matrix (80 freq bins × T time frames). |
| What does the **vocoder** output? | Raw waveform — a 1D float array at 24kHz. The actual sound. |
| How does **voice cloning** work? | Speaker embedding (192-dim vector from reference audio) conditions the flow-matching U-Net. |
| Why is it called **hybrid**? | Stage 1 (AR LLM) captures prosody/rhythm. Stage 2 (flow matching) captures acoustic detail. Best of both. |
| Why is Stage 1 slow? | Autoregressive = sequential. 2s of audio = 50 tokens generated one by one. Can't parallelize. |
| Why is Stage 2 faster? | Flow matching generates the entire mel at once (non-autoregressive), just needs ~10 iterations. |